# Gymnasium CPU Playground

**WalkingLab × Hands-On Modern RL companion experiment notebook**

Run the same curated Gymnasium recipes as the Studio, from bandits and grids to control tasks.

- Resource profile: **CPU**
- Quick run in this notebook: **2,000** training units
- Full experiment: Use each task's default budget from EXPERIMENTS for a full comparison
- [Live ModelScope Studio](https://modelscope.cn/studios/walkinglab/hands-on-modern-rl-experiment-gymnasium)
- [Experiment source](https://github.com/walkinglabs/hands-on-modern-rl/tree/main/modelscope-space/hands-on-modern-rl-experiment-gymnasium)
- [Hands-On Modern RL](https://github.com/walkinglabs/hands-on-modern-rl) · [WalkingLab](https://modelscope.cn/organization/walkinglab)

The notebook imports the exact runtime used by the Studio. Change the parameters below, run the cells in order,
and compare the checkpoint curve with the final policy GIF or result image. The first setup can take longer because
native environments and simulator assets are cached; later runs reuse `/mnt/workspace/hands-on-modern-rl-notebooks`.


## 1. Question and run boundary

This experiment asks whether the selected policy improves on the task's evaluation metric as its training budget
increases. Start with the quick budget to verify the environment and logs. Then increase the budget only after the
complete result cell produces a curve and an artifact.

A normal ModelScope **CPU Notebook** is sufficient; no GPU is required.

A short smoke run proves that the pipeline executes; it does not prove convergence. Use the full budget above when
comparing algorithms or reporting a learned behavior.


## 2. Prepare the matching Studio runtime


In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/walkinglabs/hands-on-modern-rl.git"
SPACE_SLUG = "hands-on-modern-rl-experiment-gymnasium"
INSTALL_DEPENDENCIES = True

def locate_or_clone_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "modelscope-space" / SPACE_SLUG).is_dir():
            return candidate
    workspace = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
    target = workspace / "hands-on-modern-rl-notebooks" / "source"
    target.parent.mkdir(parents=True, exist_ok=True)
    if (target / ".git").is_dir():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(target)], check=True)
    return target

REPO_ROOT = locate_or_clone_repo()
SPACE_DIR = REPO_ROOT / "modelscope-space" / SPACE_SLUG
requirements = SPACE_DIR / "requirements.txt"
packages = SPACE_DIR / "packages.txt"
cache_root = Path("/mnt/workspace/hands-on-modern-rl-notebooks") if Path("/mnt/workspace").is_dir() else REPO_ROOT / ".cache" / "online-experiments"
cache_root.mkdir(parents=True, exist_ok=True)
digest = hashlib.sha256(requirements.read_bytes() + (packages.read_bytes() if packages.exists() else b"")).hexdigest()[:12]
marker = cache_root / f"{SPACE_SLUG}-{digest}.ready"

if INSTALL_DEPENDENCIES and not marker.exists():
    if packages.exists() and sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
        system_packages = [line.strip() for line in packages.read_text().splitlines() if line.strip() and not line.startswith("#")]
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", *system_packages], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(requirements)], check=True)
    marker.touch()
else:
    print(f"Dependency cache ready: {marker}")

os.chdir(SPACE_DIR)
if str(SPACE_DIR) not in sys.path:
    sys.path.insert(0, str(SPACE_DIR))
print(f"Repository: {REPO_ROOT}")
print(f"Experiment runtime: {SPACE_DIR}")


## 3. Choose a recipe and train


In [ ]:
import app as playground

EXPERIMENT = "Bandit · ε-greedy"
TRAINING_BUDGET = 2000
LEARNING_RATE = 0.1
GAMMA = 0.99
EPSILON = 0.10
SEED = 42

print("Curated experiments:")
for name in playground.EXPERIMENTS:
    print(" ", name)
if EXPERIMENT not in playground.EXPERIMENTS:
    raise ValueError(f"Choose EXPERIMENT from {list(playground.EXPERIMENTS)}")


In [ ]:
import html as html_module
import re
from IPython.display import Image as NotebookImage, display

results = []
for result in playground.train(EXPERIMENT, TRAINING_BUDGET, LEARNING_RATE, GAMMA, EPSILON, SEED, "English"):
    results.append(result)
    console = result[-1]
    value = getattr(console, "value", str(console))
    text = html_module.unescape(re.sub(r"<[^>]+>", "", str(value)))
    print(text[-1800:], flush=True)

if not results:
    raise RuntimeError("The playground returned no training result")
status, metric, curve, preview, artifact, console = results[-1]
display(curve)
if isinstance(preview, (str, Path)) and Path(preview).exists():
    display(NotebookImage(filename=str(preview)))
else:
    display(preview)
print("Result artifact:", getattr(artifact, "value", artifact))


## 4. Read the result before increasing the budget

Compare the first and last checkpoint values, then inspect the replay. A rising curve with an implausible replay can
indicate reward shaping, evaluation, or rendering problems. A flat quick run is also inconclusive: this notebook's
default budget is a pipeline check. For a training claim, rerun with **Use each task's default budget from EXPERIMENTS for a full comparison**, keep the seed fixed,
and compare at least three seeds before drawing a conclusion.
